# ApprovalMax v2 Great Expectations-Style Validation


In [ ]:
from datetime import datetime, timezone
import json
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

catalog = 'approvalmax_ai_platform'
gold_schema = 'gold'
monitoring_schema = 'monitoring'
gold_table = f'{catalog}.{gold_schema}.fact_approval_document_lifecycle'
result_table = f'{catalog}.{monitoring_schema}.great_expectations_results'

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{monitoring_schema}')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {result_table} (
  validation_run_id STRING, expectation_name STRING, status STRING, severity STRING,
  failed_row_count BIGINT, checked_at TIMESTAMP, details STRING
) USING DELTA''')

validation_run_id = f'gx_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}'
expectations = {
    'expect_document_id_not_null': f'SELECT * FROM {gold_table} WHERE document_id IS NULL',
    'expect_company_id_not_null': f'SELECT * FROM {gold_table} WHERE company_id IS NULL',
    'expect_document_status_accepted_values': f"SELECT * FROM {gold_table} WHERE document_status IS NULL OR document_status NOT IN ('draft','submitted','approved','rejected','paid','cancelled')",
    'expect_total_amount_non_negative': f'SELECT * FROM {gold_table} WHERE total_amount < 0',
    'expect_approval_cycle_time_non_negative': f'SELECT * FROM {gold_table} WHERE approval_cycle_time_minutes < 0',
    'expect_gold_grain_unique_by_document_id': f'SELECT document_id, count(*) AS row_count FROM {gold_table} GROUP BY document_id HAVING count(*) > 1',
}

generated_tables = [
    row.tableName
    for row in spark.sql(f'SHOW TABLES IN {catalog}.{gold_schema}').collect()
    if row.tableName.endswith('_current_dbt')
]

for table_name in generated_tables:
    full_table_name = f'{catalog}.{gold_schema}.{table_name}'
    expectation_prefix = f'expect_generated_{table_name}'
    expectations[f'{expectation_prefix}_source_table_not_null'] = f'SELECT * FROM {full_table_name} WHERE source_table IS NULL'
    expectations[f'{expectation_prefix}_sequence_id_not_null'] = f'SELECT * FROM {full_table_name} WHERE sequence_id IS NULL'
    expectations[f'{expectation_prefix}_raw_payload_not_null'] = f'SELECT * FROM {full_table_name} WHERE raw_cdc_payload IS NULL'
    expectations[f'{expectation_prefix}_primary_key_not_null'] = f'SELECT * FROM {full_table_name} WHERE primary_key IS NULL'
    expectations[f'{expectation_prefix}_grain_unique_by_primary_key'] = f'SELECT primary_key, count(*) AS row_count FROM {full_table_name} GROUP BY primary_key HAVING count(*) > 1'

schema = StructType([
    StructField('validation_run_id', StringType(), False),
    StructField('expectation_name', StringType(), False),
    StructField('status', StringType(), False),
    StructField('severity', StringType(), False),
    StructField('failed_row_count', LongType(), False),
    StructField('checked_at', TimestampType(), False),
    StructField('details', StringType(), True),
])

results = []
critical_failures = []
for name, query in expectations.items():
    failed_count = spark.sql(query).count()
    status = 'passed' if failed_count == 0 else 'failed'
    row = {
        'validation_run_id': validation_run_id,
        'expectation_name': name,
        'status': status,
        'severity': 'critical',
        'failed_row_count': failed_count,
        'checked_at': datetime.now(timezone.utc),
        'details': query,
    }
    results.append(row)
    if failed_count:
        critical_failures.append(row)

spark.createDataFrame(results, schema=schema).write.format('delta').mode('append').saveAsTable(result_table)

if critical_failures:
    raise Exception('Great Expectations validation failed: ' + json.dumps(critical_failures, default=str))

print(f'All critical validations passed. validation_run_id={validation_run_id}')
